# Ensemble on the prize sheet: HercUeNet + reader-v3 + v8-in (the three models chosen for the submission)

The three models that make the ensemble (`reader-v3 base` is predicted and shown too, for comparison only), all trained on the mix28 corpus against the
current prize sheet (`utils/mix6.py` `RUN_TAG` `_ps4`, queue `utils/queue_ps4.sh`; each model is trained twice, seeds `s1` / `s2`, `VESUVIUS_SEED_TAG`):

| model | trained by | file |
|---|---|---|
| HercUeNet (sequence model, mix28 corpus, weights after **epoch 3** of its 4-epoch schedule), maxed over the depth shifts **-2, -1, 0** | `python3 train.py` | `models/runs/cue_dropout_labelsv6_mix28_ps4_s1/epoch_3.pth` |
| reader-v3 fine-tune, mix28 corpus, **step 1000** of its 4000-step schedule, maxed over the depth shifts **-2, -1, 0** (`ENSEMBLE_*` select other steps) | `python3 finetune_readerv3.py` | `reader-v3/runs/mix28_ps4_s1/ckpt_001000.pth` |
| v8-in fine-tune, mix28 corpus, **epoch 3** of its 4-epoch schedule (the Lightning checkpoint, exported on first use), maxed over the depth shifts **-2, -1** | `python3 finetune_v8in.py` | `nader-fork/runs/v8in_mix28_4ep_ps4_s1/runs/*/checkpoints/trained-epoch2.ckpt` |
| reader-v3 base, no fine-tuning (does not contribute) | released | `reader-v3/reader-v3.pth` |

**Default selection** (`utils/mix6.py` `default_models()`): each model's map is the pixelwise max of its predictions with its depth window moved by the shifts above (HercUeNet: window offset -1 + shift; reader-v3 and v8-in: the 17 / 24 central layers moved by the shift); `ENSEMBLE_CUE_DEPTHS` / `ENSEMBLE_READER_DEPTHS` / `ENSEMBLE_V8IN_DEPTHS` change the shifts (`0` = one depth). The three checkpoints are early ones of the full schedules (4 epochs / 4000 steps / 4 epochs) that the trainers save on the way. They are looked up in the `_ps4` runs of seed `s1` first; until those exist the `_ps3` runs are used (trained against the 2026-11-09 prize sheet, so with the old exclusion zones): `models/runs/cue_dropout_labelsv6_mix28_ps3/epoch_3.pth`, `reader-v3/runs/mix28_ps3/ckpt_001000.pth` and `nader-fork/runs/v8in_mix28_4ep_ps3/runs/v8in_mix28_4ep_ps3_20261110_235217_4170d5b8/checkpoints/trained-epoch2.ckpt`. With them this notebook reproduces `output/ensemble/mix28_ps3_c3_r1000_v3_depthmax_v8dm` exactly.

The trainings made against the earlier prize sheets stay on disk: the runs tagged `_ps3` (against the 2026-11-09 sheet), the runs tagged `_ps2` (against `PRIZESHEET_NEW`), the untagged runs (against `fiber_only_20261102b_flatboi_PRIZESHEET`) and the interim
mix26 / mix18 models. They are selected with the `ENSEMBLE_*` variables, and `ENSEMBLE_SHEET=PRIZESHEET_NEW` or `ENSEMBLE_SHEET=fiber_only_20261102b_flatboi_PRIZESHEET` runs the ensemble on an earlier prize sheet. A model trained
against another prize sheet was given another exclusion zone (`utils/holdout_exclusion.py`), so it may have trained near the current prize sheet; only the `_ps4` runs are clean on it.

The prize sheet is `PRIZESHEET_11_11_flatboi_flatboi_abf` (flat alias 20261112120000, the 2026-11-11 sheet, canonical since 2026-11-12, rendered upright; the only prize sheet). None of the models trained
on it, and the trained 0846A sheets lost their papyrus near it (`utils/holdout_exclusion.py`: HercUeNet 12 vx + its full 192 px context reach, reader-v3 18 vx, v8-in 25 vx).

Each model predicts the sheet at its own resolution (v8-in and reader-v3 per pixel, the sequence model per 16 px cell, repeated up to pixels, nothing is
downsampled). Every map is then stretched over the sheet mask (a clipped linear rescale to 0..1), in one of two ways:

- **p1 -> 0, p99.5 -> 1** (HercUeNet, reader-v3 base, and the maps the product uses): the 1st percentile of the model's raw values becomes 0 and the 99.5th becomes 1.
- **p50 -> 0, p99.5 -> 1** (reader-v3 fine-tune and v8-in, `FLOOR` in the settings): the median becomes 0 and everything below it is clipped to 0. The bulk of the sheet is blank, so the median
  is the model's background level: v8-in sits on a flat pedestal (raw median 0.18, 90% of the sheet between 0.17 and 0.25) and the reader-v3 on a lower one (raw median 0.26);
  a p1 stretch turns that pedestal into grey (v8-in median 0.44). Subtracting it leaves a white background and the ink on top.

The 3x2 grid `run_ensemble(sheet)` saves (full resolution, shown at 1/10): reader-v3 base (does not contribute) | reader-v3 step 1000 (p50 -> 0) / HercUeNet epoch 3 | v8-in epoch 3 (p50 -> 0) /
**z-score composite** (mean of the per-map z-scores of the maps as shown: cue p1, reader and v8-in p50 -> 0) | **product composite** (the three maps stretched p1 -> p99.5 for all, no p50 -> 0:
high only where all three agree). Both composites are shown from their p1 to p99.5 and also saved as plain PNGs next to the grid (`ensemble_z_score.png`, `ensemble_product.png`).
The second figure `run_product_ct_2x2(sheet)` saves `product_ct_2x2.png`: product composite | ct composite view (the max projection of the 17 central layers), and the ct composite view twice more.

The code cells are setup (settings, helpers, `run_ensemble`, `run_product_ct_2x2`); the last two cells, one line each, make the 3x2 and the 2x2 figure of `SHEET`. Define the models, the sheets and the scoring in the first code cell, then run the rest. Predictions are cached under `output/ensemble/<OUTPUT_SUBDIR>/<sheet>/cache/`
(delete a file there to recompute that model); predictions of the same model name cached by any other `output/ensemble/*/` run are hard-linked in first (a cache older than the render or the checkpoint is dropped), so nothing current is recomputed. v8-in and reader-v3 run in their own virtualenvs
as subprocesses (`utils/predict_models.py`); the sequence model runs in this kernel.
v8-in reads the central 24 layers, reader-v3 the 17 central layers of its checkpoint, the sequence model a flat depth-13 surface with window offset -1 (each moved by its depth shifts, see the default selection above).


In [ ]:
import os
import sys
from pathlib import Path

REPO = Path("/vesuvius")
sys.path.insert(0, str(REPO))
from utils.mix6 import (CUE_DEPTHS, CUE_TRAIN_0846A, SHEETS_0846A, HOLDOUT, HOLDOUT_RENDER, OLD_HOLDOUT, OLD_PRIZE_SHEET, PS2_HOLDOUT, PS2_PRIZE_SHEET,
                        READER_DEPTHS, SELECTED_CUE_EPOCH, SELECTED_READER_STEP, SELECTED_V8IN_EPOCH, V8IN_DEPTHS, default_models)

# ---- sheets: name -> render zarr / mask id, and the flat-alias id (cached normalisation, stored cue map) -----------------------
# a sheet without norm_id has its flat depth-13 norm and cue computed and cached under its cache folder; mask_dir defaults to masks/
SHEETS = {
    # the PRIZE sheet (utils/mix6.py HOLDOUT, canonical since 2026-11-12), flat alias 20261112120000, never trained or validated on
    HOLDOUT_RENDER: {"zid": HOLDOUT_RENDER, "norm_id": HOLDOUT, "mask_dir": REPO / "masks" / "extras"},
    # the previous prize sheet (rotated 180 at render time, alias 20261102120000), for comparison: ENSEMBLE_SHEET=<its name>
    OLD_PRIZE_SHEET: {"zid": OLD_PRIZE_SHEET, "norm_id": OLD_HOLDOUT, "mask_dir": REPO / "masks" / "extras"},
    # the 2026-11-06 prize sheet PRIZESHEET_NEW (upright, alias 20261106120000), for comparison: ENSEMBLE_SHEET=<its name>
    PS2_PRIZE_SHEET: {"zid": PS2_PRIZE_SHEET, "norm_id": PS2_HOLDOUT, "mask_dir": REPO / "masks" / "extras"},
    # PHerc0846B segment (2026-11-10), extras-style render; no flat alias, so its norm and cue are computed and cached
    "aaaaa_abf_copy_abf_copy": {"zid": "aaaaa_abf_copy_abf_copy", "mask_dir": REPO / "masks" / "extras"},
    # the same prize sheet remade once more (2026-11-12): cue and anchor norm computed from this render
    "PRIZESHEET_11_11_flatboi_flatboi_abf_flatboi": {"zid": "PRIZESHEET_11_11_flatboi_flatboi_abf_flatboi", "mask_dir": REPO / "masks" / "extras"},
}
SHEET = os.environ.get("ENSEMBLE_SHEET", HOLDOUT_RENDER)  # the sheet of the two figures at the end; train0846a = every training sheet in turn
# the 12 PHerc0846A sheets HercUeNet trained on (flat alias norm and stored cue); the mask of each render is in masks/
TRAIN_SHEETS = []
for _alias in CUE_TRAIN_0846A:
    _render, _name = SHEETS_0846A[int(_alias)]
    SHEETS[_name] = {"zid": str(_render), "norm_id": int(_alias)}
    TRAIN_SHEETS.append(_name)
SHEET_NAMES = TRAIN_SHEETS if SHEET == "train0846a" else [SHEET]

# ---- the models, in figure order: name -> settings (kind: HercUeNet | reader_v3 | v8in, see utils/predict_models.detect_kind) --
# "in_ensemble": False = shown in the figure only, not part of the composites. The name is the cache file name: renaming recomputes.
# "floor": percentile of the model's raw map over the sheet mask that is shown (and z-scored) as 0 instead of STRETCH[0] ("p50 -> 0"); the product always uses the plain STRETCH maps.
# defaults (utils/mix6.py default_models): HercUeNet epoch 3 maxed over the depth shifts -2..0, reader-v3 step 1000 of its 4000-step schedule maxed
# over -2..0 and v8-in epoch 3 (trained-epoch2.ckpt) of its 4-epoch schedule maxed over -2..-1, of the mix28 runs of the current prize sheet
# (RUN_TAG _ps4, seed s1) or, until those exist, of the _ps3 runs (old exclusion zones; same shifts: this is the result of
# output/ensemble/mix28_ps3_c3_r1000_v3_depthmax_v8dm);
# a headless run (utils/run_notebook.py) can compare other checkpoints: ENSEMBLE_READER / ENSEMBLE_CUE / ENSEMBLE_V8IN set the paths,
# ENSEMBLE_READER_LABEL / ENSEMBLE_CUE_LABEL / ENSEMBLE_V8IN_LABEL name the step / epoch in the figure and the cache file,
# ENSEMBLE_READER_CORPUS / ENSEMBLE_CUE_CORPUS / ENSEMBLE_V8IN_CORPUS name the training corpus (e.g. mix28) in the figure and the cache file,
# ENSEMBLE_CUE_DEPTHS / ENSEMBLE_READER_DEPTHS / ENSEMBLE_V8IN_DEPTHS = the depth shifts that are maxed, e.g. "0" for one depth, "-2,-1,0"
# ENSEMBLE_SUBDIR the output folder
_DEFAULTS = default_models()
READER_CKPT = Path(os.environ.get("ENSEMBLE_READER", _DEFAULTS["reader"]))
CUE_CKPT = Path(os.environ.get("ENSEMBLE_CUE", _DEFAULTS["cue"]))
V8IN_CKPT = Path(os.environ.get("ENSEMBLE_V8IN", _DEFAULTS["v8in"]))
READER_LABEL = os.environ.get("ENSEMBLE_READER_LABEL", f"step {SELECTED_READER_STEP}")
CUE_LABEL = os.environ.get("ENSEMBLE_CUE_LABEL", f"epoch {SELECTED_CUE_EPOCH}")
V8IN_LABEL = os.environ.get("ENSEMBLE_V8IN_LABEL", f"epoch {SELECTED_V8IN_EPOCH}")
READER_CORPUS = os.environ.get("ENSEMBLE_READER_CORPUS", "mix28")
CUE_CORPUS = os.environ.get("ENSEMBLE_CUE_CORPUS", "mix28")
V8IN_CORPUS = os.environ.get("ENSEMBLE_V8IN_CORPUS", "mix28")


def _depths(variable, default):
    text = os.environ.get(variable)
    return tuple(default) if text is None else tuple(int(part) for part in text.split(",") if part.strip())


CUE_SHIFTS = _depths("ENSEMBLE_CUE_DEPTHS", CUE_DEPTHS)
READER_SHIFTS = _depths("ENSEMBLE_READER_DEPTHS", READER_DEPTHS)
V8IN_SHIFTS = _depths("ENSEMBLE_V8IN_DEPTHS", V8IN_DEPTHS)


def depth_label(label, shifts):
    """the name of a model's map: a single depth 0 is the plain label, several depths are maxed."""
    if len(shifts) == 1:
        return label if shifts[0] == 0 else f"{label}, depth {shifts[0]:+d}"
    return f"{label}, max depth {min(shifts)}..{max(shifts)}"


FLOOR = 50.0             # percentile of the raw map shown as 0 for the reader-v3 fine-tune and v8-in (their background level)

MODELS = {
    # released checkpoint, no fine-tuning
    "reader-v3 base (no fine-tuning)": {"kind": "reader_v3", "checkpoint": REPO / "reader-v3" / "reader-v3.pth", "batch_size": 4,
                                        "in_ensemble": False},
    # finetune_readerv3.py, step 1000 of its 4000-step schedule; maxed over the depth shifts READER_SHIFTS
    f"reader-v3 ({READER_CORPUS} fine-tune, {depth_label(READER_LABEL, READER_SHIFTS)})": {
        "kind": "reader_v3", "checkpoint": READER_CKPT, "batch_size": 4, "floor": FLOOR, "depths": READER_SHIFTS,
        "base_name": f"reader-v3 ({READER_CORPUS} fine-tune, {READER_LABEL})"},
    # train.py, epoch_3.pth = weights after epoch 3 of 4; flat depth-13 surface, window offset -1 + the shifts CUE_SHIFTS
    f"HercUeNet ({CUE_CORPUS}, {depth_label(CUE_LABEL, CUE_SHIFTS)})": {
        "kind": "cue_dropout", "checkpoint": CUE_CKPT, "depth_offset": -1, "infer_batch": 16, "depths": CUE_SHIFTS,
        "base_name": f"HercUeNet ({CUE_CORPUS}, {CUE_LABEL})"},
    # finetune_v8in.py, epoch 3 of its 4-epoch schedule (Lightning checkpoint, exported on first use); maxed over V8IN_SHIFTS
    f"v8-in ({V8IN_CORPUS} fine-tune, {depth_label(V8IN_LABEL, V8IN_SHIFTS)})": {
        "kind": "v8in", "batch_size": 16, "checkpoint": V8IN_CKPT, "floor": FLOOR, "depths": V8IN_SHIFTS,
        "base_name": f"v8-in ({V8IN_CORPUS} fine-tune, {V8IN_LABEL})"},
}

# ---- scoring ----------------------------------------------------------------------------------------------------------
SCORING = "z-score"      # 'z-score' | 'mean' | 'rank'  (see SCORING_METHODS below)
STRETCH = (1.0, 99.5)    # percentiles of each model's map over the sheet mask

FLAT_DEPTH = 13
EVAL_STRIDE = 32         # HercUeNet inference stride (px)
PREVIEW_DOWNSCALE = 10   # the saved figure stays full size
GRID_COLS = 2            # panels per row: reader base | reader fine-tune / HercUeNet | v8-in / z-score composite | product composite
OUTPUT_SUBDIR = os.environ.get("ENSEMBLE_SUBDIR", "mix28_ps4_c3_r1000_v3_s1")  # output/ensemble/<OUTPUT_SUBDIR>/<sheet>/ ; keeps these runs apart from the earlier ensembles


In [ ]:
import gc
import os
import shutil
import types

import cv2
import numpy as np
import torch
import zarr
from IPython.display import display
from PIL import Image

# nonml_ink.py (imported by utils.precomputed_asf) lives in old/
for _path in (REPO, REPO / "old"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))
os.chdir(REPO)
Image.MAX_IMAGE_PIXELS = None

from utils import predict_models
from utils.mix6 import zarr_mtime
from utils.fast_figure import colorize, orient_display, save_panel_grid

ENSEMBLE_ROOT = REPO / "output" / "ensemble" / OUTPUT_SUBDIR


def slug(text):
    return "".join(ch if ch.isalnum() else "_" for ch in text).strip("_")


class Sheet:
    def __init__(self, name):
        spec = SHEETS[name]
        self.name, self.zid, self.norm_id = name, spec["zid"], spec.get("norm_id")
        self.norm_from = spec.get("norm_from")  # a cached norm borrowed from another sheet; the cue is still computed from this render
        self.mask_dir = Path(spec.get("mask_dir", REPO / "masks"))
        self.zarr_path = REPO / "ves_zarrs2" / f"{self.zid}.zarr"
        self.out_dir = ENSEMBLE_ROOT / name
        self.cache = self.out_dir / "cache"
        self.cache.mkdir(parents=True, exist_ok=True)
        self.mask = cv2.imread(str(self.mask_dir / f"{self.zid}.png"), cv2.IMREAD_GRAYSCALE) > 0
        self.frame = self.mask.shape

    def to_frame(self, array):
        # map of any extent -> float32 frame, NaN outside the sheet mask
        out = np.full(self.frame, np.nan, np.float32)
        rows, cols = min(self.frame[0], array.shape[0]), min(self.frame[1], array.shape[1])
        out[:rows, :cols] = array[:rows, :cols]
        out[~self.mask] = np.nan
        return out


In [ ]:
# ---- v8-in and reader-v3: subprocesses in their own virtualenvs (utils/predict_models.py) ------------------------------------
def run_v8in(sheet, name, spec):
    # the central 24 layers of the render are staged in a scratch folder (removed afterwards) when the prediction is not cached
    return predict_models.predict_v8in(sheet.zarr_path, spec["checkpoint"], sheet.cache / slug(name),
                                       batch_size=spec.get("batch_size", 16))


def run_reader_v3(sheet, name, spec):
    return predict_models.predict_reader(sheet.zarr_path, spec["checkpoint"], sheet.cache / f"{slug(name)}.tif",
                                         batch_size=spec.get("batch_size", 4))


def flat_anchor_norm(sheet, surface_depth, surface_confidence, volume=None):
    """surface-anchor norm of a sheet without a flat alias, from its flat depth-13 surface (as 0846A_eval.ipynb flat_norm)."""
    import json
    import tempfile
    from utils.norm import ANCHOR_GAP_LEVEL, ANCHOR_PAPYRUS_LEVEL, compute_surface_anchors

    path = sheet.cache / "flat_anchor_norm.json"
    if path.is_file() and path.stat().st_mtime >= zarr_mtime(sheet.zarr_path):
        return tuple(json.loads(path.read_text())["norm"])
    with tempfile.TemporaryDirectory(dir=sheet.cache) as temporary:
        surface_dir = Path(temporary) / str(sheet.zid)
        surface_dir.mkdir()
        np.save(surface_dir / "depth.npy", surface_depth)
        np.save(surface_dir / "confidence.npy", surface_confidence)
        anchors = compute_surface_anchors(sheet.zid, str(sheet.zarr_path.parent), surface_dir=temporary,
                                          mask_dir=str(sheet.mask_dir), volume=volume)
    span = (anchors["papyrus"] - anchors["gap"]) / (ANCHOR_PAPYRUS_LEVEL - ANCHOR_GAP_LEVEL)
    lower = anchors["gap"] - ANCHOR_GAP_LEVEL * span
    norm = (0.0, 1.0, lower, lower + span)
    path.write_text(json.dumps({"norm": norm, "anchors": anchors}, indent=2))
    return norm


# ---- HercUeNet: in this kernel, replaying the checkpoint's run config as 0846A_eval.ipynb does ----------------------------
def run_cue_dropout(sheet, name, spec):
    path = sheet.cache / f"{slug(name)}.npz"
    # a regenerated sheet or a retrained checkpoint is newer than the cached prediction
    if not path.is_file() or path.stat().st_mtime < max(zarr_mtime(sheet.zarr_path), Path(spec["checkpoint"]).stat().st_mtime):
        from utils.config import Config, ScrollConfig, resolve_run_config
        from utils.model import create_model
        from utils.norm import UNIFIED_CACHE_PATH, load_cached_norm
        from utils.platform import get_zarr_dir
        from utils.visualizer import TensorboardVisualizer, predict_tiles

        checkpoint = Path(spec["checkpoint"]).resolve()
        run_config, _ = resolve_run_config(checkpoint)
        C = Config()
        for section in ("data", "model"):
            for key, value in run_config[section].items():
                if section == "data" and key == "scrolls":
                    C.data.scrolls = [ScrollConfig(**item) for item in value]
                elif key != "probe_rois" and hasattr(getattr(C, section), key):
                    setattr(getattr(C, section), key, value)
        C.model.compile_model = False
        C.tra.supcon = False
        C.tra.dann = False
        C.device = "cuda" if torch.cuda.is_available() else "cpu"
        C.data.eval_stride = EVAL_STRIDE
        C.data.eval_infer_bs = int(spec.get("infer_batch", 0))  # 0 = automatic; a small value saves VRAM, results are identical
        offset = int(spec.get("depth_offset", -1))
        C.data.surface_window_offset = offset
        C.data.surface_window_offset_by_scroll = {}
        C.data.zarr_path = str(Path(get_zarr_dir()))
        depth = int(C.data.depth)
        if depth != 8 or not C.model.surface_teacher_input or not C.data.surface_relative_depth_window:
            raise ValueError("Select a literal-surface, eight-slice checkpoint")
        model, _ = create_model(C)
        state = torch.load(checkpoint, map_location="cpu", weights_only=True)
        state = state.get("model_state_dict", state.get("state_dict", state))
        state = {key.removeprefix("module.").removeprefix("_orig_mod."): value for key, value in state.items()}
        state = {key: value for key, value in state.items() if not key.startswith(("supcon_head.", "domain_head."))}
        model.load_state_dict(state, strict=True)
        model.eval()

        volume = predict_models.read_volume(predict_models.local_zarr(sheet.zarr_path))
        volume.setflags(write=False)
        height, width = sheet.mask.shape
        surface_depth = np.where(sheet.mask, FLAT_DEPTH, 255).astype(np.uint8)
        surface_confidence = np.where(sheet.mask, 255, 0).astype(np.uint8)
        if spec.get("surface_dir"):  # a predicted surface (utils/generate_surface_supervision) instead of the flat depth 13
            surface_depth = np.load(Path(spec["surface_dir"]) / "depth.npy")
            surface_confidence = np.load(Path(spec["surface_dir"]) / "confidence.npy")
        if sheet.norm_id is not None or sheet.norm_from is not None:
            norm_key = sheet.norm_from if sheet.norm_from is not None else sheet.norm_id
            norm = load_cached_norm(norm_key, UNIFIED_CACHE_PATH, mode=str(C.data.norm_mode))
            if norm is None:
                raise RuntimeError(f"no cached normalisation for {norm_key}")
        elif str(C.data.norm_mode) in ("surface_anchor", "surface_anchor_knee"):
            norm = flat_anchor_norm(sheet, surface_depth, surface_confidence, volume)
        else:
            raise ValueError(f"{sheet.name} has no flat alias; norm mode {C.data.norm_mode} needs a norm_id")
        cue = None
        if str(C.model.input_cue) == "pre_clahe":
            if sheet.norm_id is not None:
                cue = np.load(Path(C.data.cue_dir) / f"{sheet.norm_id}.npy", mmap_mode="r")
            else:
                from utils.precomputed_asf import cue_from_window

                start = FLAT_DEPTH - (depth - 1) // 2 + offset
                cue_path = sheet.cache / f"cue_s{start}.npy"
                if not cue_path.is_file() or cue_path.stat().st_mtime < zarr_mtime(sheet.zarr_path):
                    np.save(cue_path, cue_from_window(np.asarray(volume[start:start + depth], dtype=np.float32), sheet.mask))
                cue = np.load(cue_path, mmap_mode="r")
        coords = TensorboardVisualizer._gen_tile_coords(
            types.SimpleNamespace(c=C), (C.data.d_start, C.data.d_end), (0, height), (0, width), sheet.mask, z_step=depth,
        )
        grid = predict_tiles(
            C, model, volume, sheet.mask, coords, (0, height), (0, width), C.data.d_start, sheet.name, *norm,
            surface_depth_map=surface_depth, surface_confidence_map=surface_confidence,
            surface_depth_offset=offset, cue_map=cue,
        )
        path.unlink(missing_ok=True)  # the cache may be a hard link to another run's file
        np.savez(path, grid=np.asarray(grid, dtype=np.float32), cell=int(C.model.multitile_subtile))
        del model, volume
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    saved = np.load(path)
    cell = int(saved["cell"])
    return np.repeat(np.repeat(saved["grid"], cell, axis=0), cell, axis=1)


# ---- the pixelwise max over depth shifts (spec["depths"]; (0,) = the plain runner) -----------------------------------------------
def sub_name(base, shift):
    """the name of the map of one depth shift: the base name for shift 0, else `<base without its last bracket>, depth <shift>)`."""
    return base if shift == 0 else f"{base[:-1]}, depth {shift:+d})"


def newest_input(sheet, spec):
    return max(zarr_mtime(sheet.zarr_path), predict_models.checkpoint_mtime(spec["checkpoint"]))


def prepare_cue_max(sheet, name, spec, shifts):
    path = sheet.cache / f"{slug(name)}.npz"
    if path.is_file() and path.stat().st_mtime >= newest_input(sheet, spec):
        return
    grids, cell = [], None
    for shift in shifts:
        sub = sub_name(spec["base_name"], shift)
        run_cue_dropout(sheet, sub, {**spec, "depth_offset": int(spec.get("depth_offset", -1)) + shift})
        saved = np.load(sheet.cache / f"{slug(sub)}.npz")
        grids.append(saved["grid"])
        cell = int(saved["cell"])
    path.unlink(missing_ok=True)
    np.savez(path, grid=np.fmax.reduce(grids).astype(np.float32), cell=cell)


def prepare_reader_max(sheet, name, spec, shifts):
    tif = sheet.cache / f"{slug(name)}.tif"
    npy = tif.with_suffix(".npy")
    if tif.is_file() and npy.is_file() and min(tif.stat().st_mtime, npy.stat().st_mtime) >= newest_input(sheet, spec):
        return
    base = slug(spec["base_name"])
    maps = [predict_models.predict_reader(sheet.zarr_path, spec["checkpoint"], sheet.cache / (f"{base}.tif" if shift == 0 else f"{base}__depth{shift:+d}.tif"),
                                          batch_size=spec.get("batch_size", 4), layer_shift=shift) for shift in shifts]
    result = np.clip(np.rint(np.maximum.reduce(maps) * 255.0), 0, 255).astype(np.uint8)
    tif.unlink(missing_ok=True)
    npy.unlink(missing_ok=True)
    cv2.imwrite(str(tif), result)
    np.save(npy, result)


def prepare_v8in_max(sheet, name, spec, shifts):
    folder = sheet.cache / slug(name)
    path = folder / "v8in_prediction.npy"
    if path.is_file() and path.stat().st_mtime >= newest_input(sheet, spec):
        return
    base = slug(spec["base_name"])
    maps = [predict_models.predict_v8in(sheet.zarr_path, spec["checkpoint"], sheet.cache / (base if shift == 0 else f"{base}__depth{shift:+d}"),
                                        batch_size=spec.get("batch_size", 16), shift=shift) for shift in shifts]
    folder.mkdir(exist_ok=True)
    path.unlink(missing_ok=True)
    np.save(path, np.maximum.reduce(maps).astype(np.float16))


PREPARE_MAX = {"cue_dropout": prepare_cue_max, "reader_v3": prepare_reader_max, "v8in": prepare_v8in_max}


def with_depths(kind, runner):
    def run(sheet, name, spec):
        shifts = tuple(spec.get("depths", (0,)))
        if shifts != (0,):
            PREPARE_MAX[kind](sheet, name, spec, shifts)
            print(f"    {name}: max over the depth shifts {shifts}")
        return runner(sheet, name, spec)

    return run


RUNNERS = {kind: with_depths(kind, runner) for kind, runner in
           (("cue_dropout", run_cue_dropout), ("reader_v3", run_reader_v3), ("v8in", run_v8in))}


In [ ]:
# ---- stretching and scoring ----------------------------------------------------------------------------------------------
def stretch(prediction, mask):
    valid = mask & np.isfinite(prediction)
    low, high = np.percentile(prediction[valid], STRETCH)
    stretched = np.clip((prediction - low) / max(float(high - low), 1e-6), 0, 1)
    return np.where(valid, stretched, np.nan).astype(np.float32), float(low), float(high)


def stretch_from(prediction, mask, base):
    # raw percentile `base` (the background level of the map) -> 0, STRETCH[1] -> 1
    valid = mask & np.isfinite(prediction)
    low, high = np.percentile(prediction[valid], (base, STRETCH[1]))
    stretched = np.clip((prediction - low) / max(float(high - low), 1e-6), 0, 1)
    return np.where(valid, stretched, np.nan).astype(np.float32), float(low), float(high)


def link_cached(sheet, names):
    """hard-link the cached predictions of these models from any other run (output/ensemble/*/<sheet>/cache) into this run's cache."""
    for name in names:
        newest = max(zarr_mtime(sheet.zarr_path), predict_models.checkpoint_mtime(names[name]["checkpoint"]))
        for source in sorted(ENSEMBLE_ROOT.parent.glob(f"*/{sheet.name}/cache/{slug(name)}*")):
            target = sheet.cache / source.name
            if source.parent == sheet.cache or target.exists():
                continue
            # a prediction of an older training with the same name is not this checkpoint's
            if (source / "v8in_prediction.npy" if source.is_dir() else source).stat().st_mtime < newest:
                continue
            if source.is_dir():
                shutil.copytree(source, target, copy_function=os.link)
            else:
                os.link(source, target)
            print(f"linked {source}")


def z_score(values):
    valid = np.isfinite(values)
    return ((values - values[valid].mean()) / max(float(values[valid].std()), 1e-6)).astype(np.float32)


def rank_score(values):
    valid = np.isfinite(values)
    ranks = np.full(values.shape, np.nan, np.float32)
    order = np.argsort(values[valid], kind="stable")
    scores = np.empty(order.size, np.float32)
    scores[order] = np.linspace(0, 1, order.size, dtype=np.float32)
    ranks[valid] = scores
    return ranks


# stretched maps (list) -> ensemble map; every method returns NaN outside the sheet
SCORING_METHODS = {
    "z-score": lambda maps: np.mean([z_score(m) for m in maps], axis=0),
    "mean": lambda maps: np.mean(maps, axis=0),
    "rank": lambda maps: np.mean([rank_score(m) for m in maps], axis=0),
}
if SCORING not in SCORING_METHODS:
    raise ValueError(f"SCORING must be one of {sorted(SCORING_METHODS)}")

RED = (220, 0, 0)


def run_ensemble(sheet_name):
    # run every model on the sheet (cached predictions are reused), stretch, combine the models with in_ensemble != False:
    # z-score composite of the maps as shown (p50 -> 0 for models with a "floor") and product composite of the plain p1-p99.5 maps;
    # save the 3x2 grid at full resolution plus both composites as plain PNGs
    sheet = Sheet(sheet_name)
    print(f"{sheet.name}: frame {sheet.frame}, {int(sheet.mask.sum()):,} papyrus pixels; cache {sheet.cache}")
    link_cached(sheet, MODELS)
    plain, shown, limits = {}, {}, {}
    for name, spec in MODELS.items():
        print(f"[{spec['kind']}] {name}: {spec['checkpoint']}")
        if predict_models.detect_kind(spec["checkpoint"]) != spec["kind"]:
            raise ValueError(f"{spec['checkpoint']} is not a {spec['kind']} model")
        raw = sheet.to_frame(RUNNERS[spec["kind"]](sheet, name, spec))
        plain[name], low, high = stretch(raw, sheet.mask)
        floor = spec.get("floor")
        if floor is None:
            shown[name] = plain[name]
        else:
            shown[name], low, high = stretch_from(raw, sheet.mask, floor)
        limits[name] = (STRETCH[0] if floor is None else floor, low, high)
        print(f"    p{limits[name][0]:g} {low:.4f} -> 0, p{STRETCH[1]:g} {high:.4f} -> 1")
        del raw
        gc.collect()

    members = [name for name, spec in MODELS.items() if spec.get("in_ensemble", True)]
    composites = {
        f"{SCORING}": SCORING_METHODS[SCORING]([shown[name] for name in members]),
        # product of the plain p1-p99.5 maps (no p50 -> 0) of the three models: high only where all of them agree
        "product": np.prod([plain[name] for name in members], axis=0).astype(np.float32),
    }
    bad = (0.78, 0.86, 0.95)
    composite_rgb = {}
    for label, array in composites.items():
        valid = np.isfinite(array)
        c_low, c_high = np.percentile(array[valid], STRETCH)
        print(f"{label} composite of {len(members)}: p{STRETCH[0]:g} {c_low:.3f}, p{STRETCH[1]:g} {c_high:.3f}")
        composite_rgb[label] = colorize(orient_display(array, sheet.name), "gray_r", bad, vmin=float(c_low), vmax=float(c_high))
        np.save(sheet.out_dir / f"ensemble_{slug(label)}.npy", array.astype(np.float16))
        png = sheet.out_dir / f"ensemble_{slug(label)}.png"
        cv2.imwrite(str(png), composite_rgb[label][..., ::-1], [cv2.IMWRITE_PNG_COMPRESSION, 1])
        print(f"Saved {png}")

    panels = []
    for name, array in shown.items():
        base, low, high = limits[name]
        title = f"{name} | raw p{base:g}->0, p{STRETCH[1]:g}->1"
        if not MODELS[name].get("in_ensemble", True):
            title = (f"{title} | DOES NOT CONTRIBUTE", RED)
        panels.append((title, colorize(orient_display(array, sheet.name), "gray_r", bad), cv2.INTER_AREA, None))
    panels.append((f"{SCORING} composite of {len(members)} models | maps as shown above",
                   composite_rgb[SCORING], cv2.INTER_AREA, None))
    panels.append((f"product composite of {len(members)} models | all maps p{STRETCH[0]:g}->0, no p50->0",
                   composite_rgb["product"], cv2.INTER_AREA, None))
    cell_hw = composite_rgb[SCORING].shape[:2]

    rows = [panels[i:i + GRID_COLS] for i in range(0, len(panels), GRID_COLS)]
    grid_path = sheet.out_dir / f"ensemble_{slug(SCORING)}_{len(rows)}x{GRID_COLS}.png"
    save_panel_grid(rows, cell_hw, str(grid_path), gap_px=24)
    print(f"Saved {grid_path}")
    with Image.open(grid_path) as saved:
        print("full size", saved.size)
        display(saved.resize((max(1, saved.width // PREVIEW_DOWNSCALE), max(1, saved.height // PREVIEW_DOWNSCALE)),
                             Image.Resampling.LANCZOS))
    del plain, shown, composites, composite_rgb, panels
    gc.collect()
    return grid_path


In [ ]:
# ---- the simple 2x2 figure: product composite | ct composite view, and the ct composite view twice more ----------------------
from utils.fast_figure import gray_rgb


def ct_composite(sheet):
    """masked max projection of the 17 central layers reader-v3 reads (the composite panel of 0846A_eval.ipynb)."""
    volume = zarr.open(str(sheet.zarr_path), mode="r")
    start, end = predict_models.native_window("reader_v3", volume.shape[0], MODELS["reader-v3 base (no fine-tuning)"]["checkpoint"])
    composite = np.zeros(sheet.frame, np.uint8)
    for y in range(0, sheet.frame[0], 512):
        composite[y:y + 512] = np.clip(np.asarray(volume[start:end, y:y + 512, :sheet.frame[1]]).max(axis=0), 0, 255)
    return composite * sheet.mask, start, end


def run_product_ct_2x2(sheet_name):
    # needs run_ensemble(sheet_name) first: reads its ensemble_product.png
    sheet = Sheet(sheet_name)
    product_rgb = cv2.imread(str(sheet.out_dir / "ensemble_product.png"))[..., ::-1]  # saved by run_ensemble (BGR on disk)
    composite, first, last = ct_composite(sheet)
    ct_rgb = gray_rgb(orient_display(composite, sheet.name))
    product_panel = ("3-model product composite", product_rgb, cv2.INTER_AREA, None)
    ct_panel = (f"ct composite view (layers {first}-{last - 1})", ct_rgb, cv2.INTER_AREA, 9.362)
    simple_path = sheet.out_dir / "product_ct_2x2.png"
    save_panel_grid([[product_panel, ct_panel], [ct_panel, ct_panel]], ct_rgb.shape[:2], str(simple_path), gap_px=24)
    print(f"Saved {simple_path}")
    with Image.open(simple_path) as saved:
        display(saved.resize((max(1, saved.width // PREVIEW_DOWNSCALE), max(1, saved.height // PREVIEW_DOWNSCALE)), Image.Resampling.LANCZOS))
    del composite, ct_rgb, product_rgb
    gc.collect()
    return simple_path


In [ ]:
prize_sheet_paths = [run_ensemble(name) for name in SHEET_NAMES]

In [ ]:
product_ct_paths = [run_product_ct_2x2(name) for name in SHEET_NAMES]